# Phase 1 (0.6B variant): Convert fine-tuned Qwen3-0.6B LoRA adapter to GGUF (Q4_K_M)

This is the 0.6B sibling of `convert_gguf_qwen3_1p7b.ipynb`. The 1.7B path stays the production one; this notebook exists so we can A/B reliability + on-device latency against the smaller model.

**What this notebook does**

1. Loads the **non-quantized** base `Qwen/Qwen3-0.6B` (NOT the bnb-4bit base; merging a LoRA into a 4-bit base for GGUF conversion is brittle).
2. Applies the LoRA adapter from `unsloth_qwen3_0p6b_parser_run/lora_adapter/`.
3. Merges the LoRA into the base weights and saves a full-precision HF snapshot.
4. Clones llama.cpp and runs `convert_hf_to_gguf.py` to produce a single F16 GGUF.
5. Quantizes that GGUF to **Q4_K_M** (~400 MB), the format the Android app will load.
6. Sanity-tests the produced GGUF with `llama-cpp-python` against one parser prompt.

**Inputs you must provide via `CONFIG` cell below:**
- The LoRA adapter folder produced by `finetune_qwen3_0p6b.py`
- HF token if `Qwen/Qwen3-0.6B` requires gated access (it doesn't currently, but kept as env var for safety)

**Output you take back to your machine:**
- One file: `qwen3-0.6b-parser-q4_k_m.gguf` (~400 MB). Push it to the same `/sdcard/Android/data/com.secondbrain.app/files/models/` folder as the existing 1.7B GGUF; the app's Settings screen will list both and let you choose.

**Hardware:** No GPU strictly required for conversion. T4 free tier is fine and faster than CPU. ~4 GB scratch disk used during conversion (vs ~10 GB for 1.7B).

## CONFIG â€” edit these and run the rest top-to-bottom

In [ ]:
import os
from pathlib import Path

# --- Inputs ---------------------------------------------------------------
# Kaggle: adapter folder is attached as an input dataset.
# Colab: point this at the LoRA adapter produced by finetune_qwen3_0p6b.py.
# The folder must contain: adapter_config.json, adapter_model.safetensors, tokenizer.json, tokenizer_config.json
ADAPTER_DIR = "/content/drive/MyDrive/notes_app_finetuning/unsloth_qwen3_0p6b_parser_run/lora_adapter"

# Base model that the LoRA was trained against. Use the NON-quantized HF repo for clean merge â†’ GGUF.
# Verify by reading adapter_config.json in your adapter folder â€” unsloth/Qwen3-0.6B-unsloth-bnb-4bit was the
# training base, but its non-quantized counterpart is `Qwen/Qwen3-0.6B`.
BASE_MODEL_HF = "Qwen/Qwen3-0.6B"

# --- Outputs --------------------------------------------------------------
WORK_DIR = Path("/kaggle/working/work")  # change to /content/work on Colab
MERGED_DIR = WORK_DIR / "merged_hf"           # full-precision HF after LoRA merge
LLAMA_CPP_DIR = WORK_DIR / "llama.cpp"         # cloned llama.cpp for conversion + quantize
F16_GGUF = WORK_DIR / "qwen3-0.6b-parser-f16.gguf"
Q4_GGUF = WORK_DIR / "qwen3-0.6b-parser-q4_k_m.gguf"

WORK_DIR.mkdir(parents=True, exist_ok=True)

# --- Optional HF token (only needed if base model becomes gated) ----------
# os.environ["HF_TOKEN"] = "hf_xxx"

print(f"ADAPTER_DIR  = {ADAPTER_DIR}")
print(f"BASE_MODEL_HF = {BASE_MODEL_HF}")
print(f"WORK_DIR     = {WORK_DIR}")
print(f"Final GGUF   = {Q4_GGUF}")

## Step 0 (optional) â€” Mount Google Drive

Skip this cell if you uploaded the adapter folder directly or are running on Kaggle.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# # Then update ADAPTER_DIR above to e.g. "/content/drive/MyDrive/unsloth_qwen3_0p6b_parser_run/lora_adapter"

## Step 1 â€” Install Python dependencies

We use the upstream `transformers` + `peft` for the merge. Avoiding unsloth here so we don't fight its 4-bit-quantized base loader.

In [ ]:
%pip install -q --upgrade pip
%pip install -q "transformers>=4.46" "peft>=0.13" "safetensors>=0.4" "accelerate>=0.34" sentencepiece protobuf
%pip install -q llama-cpp-python  # used only for the sanity-test cell at the end

## Step 2 â€” Verify the adapter folder is reachable

In [ ]:
import json

adapter_path = Path(ADAPTER_DIR)
assert adapter_path.exists(), f"Adapter dir not found: {adapter_path}"

required = ["adapter_config.json", "adapter_model.safetensors", "tokenizer_config.json", "tokenizer.json"]
missing = [name for name in required if not (adapter_path / name).exists()]
assert not missing, f"Missing files in adapter dir: {missing}"

with open(adapter_path / "adapter_config.json") as f:
    adapter_cfg = json.load(f)
print("Adapter base recorded in config:", adapter_cfg.get("base_model_name_or_path"))
print("Will merge against (non-quantized): ", BASE_MODEL_HF)
print("r =", adapter_cfg.get("r"), " alpha =", adapter_cfg.get("lora_alpha"))
print("target_modules =", adapter_cfg.get("target_modules"))

## Step 3 â€” Load base, apply LoRA, merge, save full-precision HF snapshot

Memory note: Qwen3-0.6B in fp16 is ~1.2 GB on GPU/RAM (vs 3.4 GB for 1.7B). Free Colab handles it easily. We then `.merge_and_unload()` to fold the LoRA deltas into the base weights so the GGUF converter sees a plain causal-LM checkpoint.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

token = os.environ.get("HF_TOKEN")

# Load tokenizer from the adapter dir (it has the chat template + special tokens) and base.
# The adapter's tokenizer is what was used during training; using it ensures the GGUF carries
# the exact chat template the Android app will rely on.
tokenizer = AutoTokenizer.from_pretrained(str(adapter_path), trust_remote_code=True, token=token)

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
device_map = "auto" if torch.cuda.is_available() else None

print(f"Loading base in {dtype} (this can take a minute on first run; weights are ~1.2 GB)â€¦")
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_HF,
    torch_dtype=dtype,
    device_map=device_map,
    trust_remote_code=True,
    token=token,
)

print("Attaching LoRA adapterâ€¦")
merged = PeftModel.from_pretrained(base, str(adapter_path), is_trainable=False)

print("Merging LoRA into base weightsâ€¦")
merged = merged.merge_and_unload()

MERGED_DIR.mkdir(parents=True, exist_ok=True)
print(f"Saving merged checkpoint to {MERGED_DIR} â€¦ (~1.2 GB on disk)")
merged.save_pretrained(str(MERGED_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(MERGED_DIR))

# Free GPU/RAM before the conversion step kicks in
del merged, base
import gc; gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("âœ“ merged checkpoint ready")
!ls -lh "{MERGED_DIR}"

## Step 4 â€” Clone llama.cpp and install its conversion deps

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp.git "{LLAMA_CPP_DIR}"
%pip install -q -r "{LLAMA_CPP_DIR}/requirements/requirements-convert_hf_to_gguf.txt"
!ls "{LLAMA_CPP_DIR}" | head -n 30

## Step 5 â€” Convert merged HF checkpoint to GGUF (F16)

`convert_hf_to_gguf.py` produces a `.gguf` from a Hugging Face checkpoint. We go to F16 first; the next step quantizes to Q4_K_M.

In [ ]:
!python "{LLAMA_CPP_DIR}/convert_hf_to_gguf.py" \
    "{MERGED_DIR}" \
    --outfile "{F16_GGUF}" \
    --outtype f16

!ls -lh "{F16_GGUF}"

## Step 6 â€” Build the llama.cpp `llama-quantize` binary and quantize to Q4_K_M

We only need the quantize binary, not the whole runtime, so a CPU-only minimal build is enough. Takes ~2 minutes on Colab.

In [ ]:
!apt-get -qq install -y cmake build-essential
!cmake -S "{LLAMA_CPP_DIR}" -B "{LLAMA_CPP_DIR}/build" \
    -DCMAKE_BUILD_TYPE=Release \
    -DLLAMA_CURL=OFF \
    -DGGML_NATIVE=OFF \
    -DGGML_CUDA=OFF \
    -DGGML_VULKAN=OFF \
    -DGGML_METAL=OFF \
    -DLLAMA_BUILD_SERVER=OFF \
    -DLLAMA_BUILD_TESTS=OFF \
    -DLLAMA_BUILD_EXAMPLES=ON
!cmake --build "{LLAMA_CPP_DIR}/build" --target llama-quantize -j$(nproc)

QUANTIZE_BIN = LLAMA_CPP_DIR / "build" / "bin" / "llama-quantize"
print("quantize bin:", QUANTIZE_BIN, "exists?", QUANTIZE_BIN.exists())

In [ ]:
!"{QUANTIZE_BIN}" "{F16_GGUF}" "{Q4_GGUF}" Q4_K_M
!ls -lh "{Q4_GGUF}"

## Step 7 â€” Sanity-test the Q4_K_M GGUF against one parser prompt

We run the same prompt shape that `prototypes/flask-reference/second_brain_finetuned_parser.py` builds (system prompt + user prompt rendered via the Qwen3 chat template, `enable_thinking=False`). Output should be a single JSON object with `task: parse_query`.

In [ ]:
from llama_cpp import Llama
from datetime import date

SYSTEM_PROMPT = (
    "You are a parser for a tag-first personal data app.\n"
    "Return JSON only.\n"
    "Do not add markdown.\n"
    "Do not add explanations.\n"
    "Do not add extra keys.\n"
    "Use null for missing values.\n"
    "Follow the schema shown by the examples exactly."
)
today_iso = date.today().isoformat()
system_full = f"{SYSTEM_PROMPT}\n\nToday: {today_iso}"

USER_PROMPT = "ask: latest buy list"

# Render with the tokenizer's chat template so the prompt matches training
messages = [
    {"role": "system", "content": system_full},
    {"role": "user", "content": USER_PROMPT},
]
chat_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
print("--- chat template rendered prompt ---\n" + chat_text)

llm = Llama(
    model_path=str(Q4_GGUF),
    n_ctx=1024,
    n_threads=os.cpu_count() or 2,
    n_gpu_layers=0,  # CPU sanity test; on Pixel 7 the Android app will offload to Vulkan
    verbose=False,
)

out = llm(
    chat_text,
    max_tokens=256,
    temperature=0.0,
    top_p=1.0,
    stop=["<|im_end|>"],
)
raw = out["choices"][0]["text"].strip()
print("\n--- raw model output ---\n" + raw)

import json as _json
try:
    parsed = _json.loads(raw)
    print("\nâœ“ Output parses as JSON:")
    print(_json.dumps(parsed, indent=2))
except Exception as exc:
    print(f"\nâœ— Output is not valid JSON: {exc}")
    print("This means the GGUF needs investigation before going to Phase 2.")

## Step 8 â€” Download the final GGUF

On Colab, the easiest path is `files.download` (will offer a browser download). On Kaggle, click the file in `/kaggle/working/` to download.

Push the downloaded file to the **same** `/sdcard/Android/data/com.secondbrain.app/files/models/` folder as the existing 1.7B GGUF. The Settings screen will list both and let you switch.

In [ ]:
try:
    from google.colab import files  # type: ignore
    files.download(str(Q4_GGUF))
except Exception as exc:
    print(f"Auto-download not available ({exc}).")
    print(f"Manually grab the file from: {Q4_GGUF}")
    !ls -lh "{Q4_GGUF}"